# 🗺️ Cartographie Sémantique — Espace Latent ECG (UMAP + Plotly)

**Objectif** : Projeter nos vecteurs OpenAI `text-embedding-3-small` (1536 dimensions) en 2D via **UMAP**,
puis tracer des cartes interactives montrant :
- 🔵 **L'ontologie ECG** (~411 documents = concepts canoniques + synonymes)
- ⭐ **Les termes étudiants** (extraits des réponses, colorés par étudiant)
- 🏗️ **Les zones sémantiques** (enveloppes convexes par catégorie)

| Cellule | Description |
|---------|-------------|
| 1 | **Imports & Setup** — numpy, umap-learn, plotly, OpenAI client |
| 2 | **Fond de Carte** — Chargement vecteurs + métadonnées ontologie |
| 3 | **Explorateurs** — Extraction & vectorisation des termes étudiants (avec lien étudiant ↔ terme) |
| 4 | **UMAP** — Réduction 1536D → 2D (fit sur ontologie, transform étudiants) |
| 5 | **Carte A** — Ontologie par catégorie + étudiants par code (dropdown par étudiant) |
| 6 | **Carte B** — Zones sémantiques (enveloppes convexes) + étudiants |
| 7 | **Distances** — Lignes étudiant → concept le + proche (similarité cosinus) |
| 8 | **Tableaux** — Meilleurs/pires matchs + histogramme de distribution |

In [1]:
# ============================================================
# CELLULE 1 — Imports & Setup
# ============================================================
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import os
import re

# UMAP pour la réduction de dimensionnalité (fallback t-SNE si besoin)
try:
    import umap
    REDUCER_NAME = 'UMAP'
    print(f"[OK] umap-learn {umap.__version__} chargé")
except ImportError:
    from sklearn.manifold import TSNE
    REDUCER_NAME = 't-SNE'
    print("[WARN] umap-learn non disponible, fallback sur t-SNE")

# Plotly pour la visualisation interactive
import plotly.express as px
import plotly.graph_objects as go

# Client OpenAI pour vectoriser les termes étudiants
from openai import OpenAI

warnings.filterwarnings('ignore', category=FutureWarning)

# ─── Chemins racines ──────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\Administrateur\bmad\ECG lecture")
EVAL_ROOT    = Path(r"C:\Users\Administrateur\bmad\ECG evaluation")
INDEX_DIR    = PROJECT_ROOT / "rag_pipeline" / "rag_index"

# Charger la clé API OpenAI
load_dotenv(PROJECT_ROOT / ".env")
client = OpenAI()

EMBEDDING_MODEL = "text-embedding-3-small"  # 1536 dims

# CSV étudiants (nom réel du fichier)
CSV_PATH = EVAL_ROOT / "ECG_Collector_Data - responses(1).csv"

print(f"[OK] OPENAI_API_KEY : {'✅' if os.getenv('OPENAI_API_KEY') else '❌'}")
print(f"[OK] Réducteur      : {REDUCER_NAME}")
print(f"[OK] Index dir      : {INDEX_DIR}")
print(f"[OK] CSV            : {CSV_PATH.name}")
print(f"\n🚀 Setup terminé.")

c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[OK] umap-learn 0.5.11 chargé
[OK] OPENAI_API_KEY : ✅
[OK] Réducteur      : UMAP
[OK] Index dir      : C:\Users\Administrateur\bmad\ECG lecture\rag_pipeline\rag_index
[OK] CSV            : ECG_Collector_Data - responses(1).csv

🚀 Setup terminé.
[OK] OPENAI_API_KEY : ✅
[OK] Réducteur      : UMAP
[OK] Index dir      : C:\Users\Administrateur\bmad\ECG lecture\rag_pipeline\rag_index
[OK] CSV            : ECG_Collector_Data - responses(1).csv

🚀 Setup terminé.


In [2]:
# ============================================================
# CELLULE 2 — Fond de Carte : Chargement de l'Ontologie
# ============================================================
# Matrice des embeddings OpenAI (N × 1536)
embeddings_onto = np.load(INDEX_DIR / "vecteurs_ontologie.npy")
print(f"[ONTO] Matrice embeddings : {embeddings_onto.shape} ({embeddings_onto.dtype})")

# Métadonnées : label, ID OWL, catégorie, type de surface form
with open(INDEX_DIR / "metadata_ontologie.json", 'r', encoding='utf-8') as f:
    meta_onto = json.load(f)

documents = meta_onto['documents']
assert len(documents) == embeddings_onto.shape[0], "Incohérence taille embeddings / métadonnées"

# Construire un DataFrame pour l'ontologie
df_onto = pd.DataFrame(documents)
df_onto['type'] = '🔵 Ontologie'
df_onto['label'] = df_onto['surface_form']
df_onto['hover'] = df_onto.apply(
    lambda r: f"{r['surface_form']}\n[{r['ontology_id']}]\n{r['categorie']} ({r['source_type']})",
    axis=1
)

# Stats
print(f"[ONTO] {len(df_onto)} documents chargés")
print(f"\n   Répartition par catégorie :")
for cat, count in df_onto['categorie'].value_counts().items():
    print(f"      {cat:<35s} : {count}")
print(f"\n   Répartition par type :")
for st, count in df_onto['source_type'].value_counts().items():
    print(f"      {st:<15s} : {count}")

print(f"\n✅ Fond de carte prêt : {len(df_onto)} points à projeter.")

[ONTO] Matrice embeddings : (537, 1536) (float32)
[ONTO] 537 documents chargés

   Répartition par catégorie :
      DESCRIPTEUR_ECG                     : 223
      DIAGNOSTIC_MAJEUR                   : 128
      SIGNE_ECG_PATHOLOGIQUE              : 126
      DIAGNOSTIC_URGENT                   : 60

   Répartition par type :
      canonical       : 292
      synonym         : 245

✅ Fond de carte prêt : 537 points à projeter.


In [3]:
# ============================================================
# CELLULE 3 — Explorateurs : Termes Étudiants Vectorisés
# ============================================================
# On garde le lien étudiant ↔ terme ↔ cas pour la carte par étudiant.

# ─── A) Charger les réponses brutes ──────────────────────────
df_responses = pd.read_csv(CSV_PATH)
PARTICIPANTS = df_responses['code'].tolist()
cas_cols = [c for c in df_responses.columns if c.startswith('cas_')]

print(f"[CSV] {len(PARTICIPANTS)} participants, {len(cas_cols)} cas")

# ─── B) Extraire les termes avec métadonnées ─────────────────
# Chaque terme garde : student_code, cas, texte brut
records = []
for _, row in df_responses.iterrows():
    code = row['code']
    for col in cas_cols:
        text = str(row[col]).strip()
        if text and text != 'nan':
            segments = re.split(r'[,\n.;]+', text)
            for seg in segments:
                seg = seg.strip()
                if 3 <= len(seg) <= 100:
                    records.append({
                        'student_code': code,
                        'cas': col,
                        'terme': seg,
                    })

df_terms = pd.DataFrame(records)

# Dédupliquer les termes identiques POUR LE MÊME étudiant (pas entre étudiants)
df_terms = df_terms.drop_duplicates(subset=['student_code', 'terme'])
print(f"[ETU] {len(df_terms)} termes (avec doublons inter-étudiants)")
print(f"   Participants : {df_terms['student_code'].nunique()}")
print(f"   Termes uniques (tous étudiants confondus) : {df_terms['terme'].nunique()}")

# ─── C) Vectoriser les termes UNIQUES (économie d'API) ───────
unique_terms = sorted(df_terms['terme'].unique())
print(f"\n⏳ Vectorisation de {len(unique_terms)} termes uniques via {EMBEDDING_MODEL}...")

BATCH_SIZE = 100
all_embeddings = []

for i in range(0, len(unique_terms), BATCH_SIZE):
    batch = unique_terms[i:i+BATCH_SIZE]
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
    batch_embs = [item.embedding for item in response.data]
    all_embeddings.extend(batch_embs)
    print(f"   Batch {i//BATCH_SIZE + 1}/{(len(unique_terms)-1)//BATCH_SIZE + 1} — {len(batch)} termes")

# Dictionnaire terme → embedding
term_to_emb = dict(zip(unique_terms, all_embeddings))

# Construire la matrice en respectant l'ordre de df_terms
embeddings_students = np.array([term_to_emb[t] for t in df_terms['terme']], dtype=np.float32)
student_terms = df_terms['terme'].tolist()  # compatibilité avec cellules suivantes

print(f"\n[ETU] Matrice embeddings : {embeddings_students.shape}")
print(f"✅ {len(unique_terms)} termes uniques vectorisés → {len(df_terms)} points (avec répétitions par étudiant).")

[CSV] 7 participants, 15 cas
[ETU] 254 termes (avec doublons inter-étudiants)
   Participants : 7
   Termes uniques (tous étudiants confondus) : 216

⏳ Vectorisation de 216 termes uniques via text-embedding-3-small...
   Batch 1/3 — 100 termes
   Batch 1/3 — 100 termes
   Batch 2/3 — 100 termes
   Batch 2/3 — 100 termes
   Batch 3/3 — 16 termes

[ETU] Matrice embeddings : (254, 1536)
✅ 216 termes uniques vectorisés → 254 points (avec répétitions par étudiant).
   Batch 3/3 — 16 termes

[ETU] Matrice embeddings : (254, 1536)
✅ 216 termes uniques vectorisés → 254 points (avec répétitions par étudiant).


In [4]:
# ============================================================
# CELLULE 4 — Réduction UMAP : 1536D → 2D
# ============================================================
# Stratégie :
#   1. fit_transform() sur l'ontologie (= espace de référence)
#   2. transform() des étudiants sur cet espace (sans le déformer)

print(f"⏳ Réduction {REDUCER_NAME} : {embeddings_onto.shape[1]}D → 2D")
print(f"   Ontologie  : {embeddings_onto.shape[0]} points")
print(f"   Étudiants  : {embeddings_students.shape[0]} points")

if REDUCER_NAME == 'UMAP':
    reducer = umap.UMAP(
        n_components=2,
        metric='cosine',
        n_neighbors=15,
        min_dist=0.1,
        random_state=42,
        verbose=True,
    )
    coords_onto = reducer.fit_transform(embeddings_onto)
    coords_students = reducer.transform(embeddings_students)
else:
    all_embs = np.vstack([embeddings_onto, embeddings_students])
    tsne = TSNE(n_components=2, metric='cosine', random_state=42, perplexity=30)
    coords_all = tsne.fit_transform(all_embs)
    coords_onto = coords_all[:len(embeddings_onto)]
    coords_students = coords_all[len(embeddings_onto):]

print(f"\n[OK] Ontologie  → {coords_onto.shape}")
print(f"[OK] Étudiants  → {coords_students.shape}")

# ─── Assembler les DataFrames ─────────────────────────────────
df_plot_onto = pd.DataFrame({
    'x': coords_onto[:, 0],
    'y': coords_onto[:, 1],
    'label': df_onto['surface_form'].values,
    'type': '🔵 Ontologie',
    'categorie': df_onto['categorie'].values,
    'source': df_onto['source_type'].values,
    'concept_id': df_onto['ontology_id'].values,
    'student_code': '',
    'cas': '',
})

df_plot_students = pd.DataFrame({
    'x': coords_students[:, 0],
    'y': coords_students[:, 1],
    'label': df_terms['terme'].values,
    'type': '⭐ Étudiant',
    'categorie': 'RÉPONSE_ÉTUDIANT',
    'source': 'student',
    'concept_id': '',
    'student_code': df_terms['student_code'].values,
    'cas': df_terms['cas'].values,
})

df_plot = pd.concat([df_plot_onto, df_plot_students], ignore_index=True)
print(f"\n✅ DataFrame combiné : {len(df_plot)} points ({len(df_plot_onto)} onto + {len(df_plot_students)} étudiants)")
print(f"   Étudiants par code :")
for code, n in df_plot_students['student_code'].value_counts().items():
    print(f"      {code} : {n} termes")

⏳ Réduction UMAP : 1536D → 2D
   Ontologie  : 537 points
   Étudiants  : 254 points
UMAP(angular_rp_forest=True, metric='cosine', n_jobs=1, random_state=42, verbose=True)
Sat Feb 28 23:52:59 2026 Construct fuzzy simplicial set


c:\Python314\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Sat Feb 28 23:52:59 2026 Finding Nearest Neighbors
Sat Feb 28 23:53:02 2026 Finished Nearest Neighbor Search
Sat Feb 28 23:53:02 2026 Finished Nearest Neighbor Search
Sat Feb 28 23:53:03 2026 Construct embedding
Sat Feb 28 23:53:03 2026 Construct embedding


Epochs completed:  30%| ███        152/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs


Epochs completed: 100%| ██████████ 500/500 [00:00]



	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Sat Feb 28 23:53:04 2026 Finished embedding


Epochs completed: 100%| ██████████ 100/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs

[OK] Ontologie  → (537, 2)
[OK] Étudiants  → (254, 2)

✅ DataFrame combiné : 791 points (537 onto + 254 étudiants)
   Étudiants par code :
      ECG-IDXQ : 84 termes
      ECG-3RMP : 52 termes
      ECG-DFLC : 33 termes
      ECG-1I3Q : 33 termes
      ECG-WY55 : 31 termes
      ECG-7512 : 16 termes
      ECG-7WHN : 5 termes


In [5]:
# ============================================================
# CELLULE 5 — Carte A : Ontologie par Catégorie + Étudiants par Code
# ============================================================
# Ontologie colorée par catégorie (fond) + étudiants colorés par code
# avec dropdown pour isoler un étudiant ou voir tout le monde.

COLOR_MAP_CAT = {
    'DIAGNOSTIC_URGENT':        '#F44336',
    'DIAGNOSTIC_MAJEUR':        '#2196F3',
    'SIGNE_ECG_PATHOLOGIQUE':   '#FF9800',
    'DESCRIPTEUR_ECG':          '#4CAF50',
}

# Palette distincte pour chaque étudiant
STUDENT_COLORS = [
    '#E91E63', '#9C27B0', '#00BCD4', '#CDDC39', '#FF5722',
    '#3F51B5', '#009688', '#FFC107', '#795548', '#607D8B',
]
student_codes = sorted(df_plot_students['student_code'].unique())
color_map_stu = {code: STUDENT_COLORS[i % len(STUDENT_COLORS)] for i, code in enumerate(student_codes)}

fig = go.Figure()

# ─── A) Fond : Ontologie par catégorie ────────────────────────
for cat, color in COLOR_MAP_CAT.items():
    mask = df_plot_onto['categorie'] == cat
    subset = df_plot_onto[mask]
    if len(subset) == 0:
        continue
    fig.add_trace(go.Scatter(
        x=subset['x'], y=subset['y'],
        mode='markers',
        marker=dict(size=5, color=color, opacity=0.35, line=dict(width=0.3, color='#555')),
        name=f"Onto: {cat.replace('_', ' ').title()}",
        text=subset['label'],
        customdata=np.stack([subset['concept_id'], subset['source']], axis=-1),
        hovertemplate="<b>%{text}</b><br>ID: %{customdata[0]}<br><extra>%{customdata[1]}</extra>",
        legendgroup='onto',
    ))

# ─── B) Étudiants par code (chacun sa couleur + son trace) ───
for code in student_codes:
    mask = df_plot_students['student_code'] == code
    subset = df_plot_students[mask]
    fig.add_trace(go.Scatter(
        x=subset['x'], y=subset['y'],
        mode='markers',
        marker=dict(
            size=10, symbol='star',
            color=color_map_stu[code],
            opacity=0.9,
            line=dict(width=1, color='#333'),
        ),
        name=f"⭐ {code}",
        text=subset['label'],
        customdata=subset['cas'].values,
        hovertemplate="<b>⭐ %{text}</b><br>Étudiant: " + code + "<br>Cas: %{customdata}<extra></extra>",
        legendgroup='students',
    ))

# ─── C) Dropdown pour filtrer par étudiant ────────────────────
n_onto_traces = len(COLOR_MAP_CAT)  # 4 traces ontologie
n_students = len(student_codes)

def make_visibility(selected_idx=None):
    """Retourne la visibilité : ontologie toujours visible, étudiants filtrés."""
    vis = [True] * n_onto_traces  # Ontologie toujours visible
    for i in range(n_students):
        vis.append(True if selected_idx is None else (i == selected_idx))
    return vis

buttons = [dict(label="👥 Tous les étudiants", method="update",
                args=[{"visible": make_visibility(None)}])]
for i, code in enumerate(student_codes):
    buttons.append(dict(
        label=f"⭐ {code}",
        method="update",
        args=[{"visible": make_visibility(i)}],
    ))

fig.update_layout(
    updatemenus=[dict(
        type="dropdown", direction="down",
        x=0.02, y=0.98, xanchor="left", yanchor="top",
        bgcolor='#2d2d2d', bordercolor='#666', font=dict(color='#e0e0e0', size=11),
        buttons=buttons,
    )],
    title=dict(
        text="🗺️ Cartographie Sémantique — Ontologie ECG vs Réponses par Étudiant",
        font=dict(size=17, color='#e0e0e0'), x=0.5,
    ),
    xaxis=dict(title=f"{REDUCER_NAME} Dim 1", showgrid=True, gridcolor='#333', zeroline=False),
    yaxis=dict(title=f"{REDUCER_NAME} Dim 2", showgrid=True, gridcolor='#333', zeroline=False),
    plot_bgcolor='#1e1e1e', paper_bgcolor='#1e1e1e',
    font=dict(color='#e0e0e0'),
    legend=dict(bgcolor='#2d2d2d', bordercolor='#444', borderwidth=1, font=dict(size=11)),
    width=1200, height=800, hovermode='closest',
)

fig.show()

print(f"\n📊 Carte par étudiant tracée : {len(df_plot_onto)} points ontologie + {len(df_plot_students)} termes étudiants")
print(f"   🎛️ Utilisez le dropdown en haut à gauche pour isoler un étudiant.")
print(f"   💡 Survolez les ⭐ pour voir le terme et le cas ECG.")


📊 Carte par étudiant tracée : 537 points ontologie + 254 termes étudiants
   🎛️ Utilisez le dropdown en haut à gauche pour isoler un étudiant.
   💡 Survolez les ⭐ pour voir le terme et le cas ECG.


In [6]:
# ============================================================
# CELLULE 5bis — Carte B : Zones Sémantiques (Enveloppes Convexes)
# ============================================================
# Dessine les enveloppes convexes de chaque catégorie ontologique
# pour identifier les territoires sémantiques sur la carte.

from scipy.spatial import ConvexHull

fig_zones = go.Figure()

# ─── A) Enveloppes convexes par catégorie ─────────────────────
ZONE_COLORS = {
    'DIAGNOSTIC_URGENT':        ('#F44336', 'rgba(244,67,54,0.08)'),
    'DIAGNOSTIC_MAJEUR':        ('#2196F3', 'rgba(33,150,243,0.08)'),
    'SIGNE_ECG_PATHOLOGIQUE':   ('#FF9800', 'rgba(255,152,0,0.08)'),
    'DESCRIPTEUR_ECG':          ('#4CAF50', 'rgba(76,175,80,0.08)'),
}

ZONE_LABELS = {
    'DIAGNOSTIC_URGENT':        '🔴 Urgences',
    'DIAGNOSTIC_MAJEUR':        '🔵 Diagnostics majeurs',
    'SIGNE_ECG_PATHOLOGIQUE':   '🟠 Signes pathologiques',
    'DESCRIPTEUR_ECG':          '🟢 Descripteurs',
}

for cat, (line_color, fill_color) in ZONE_COLORS.items():
    mask = df_plot_onto['categorie'] == cat
    subset = df_plot_onto[mask]
    if len(subset) < 3:
        continue

    points = subset[['x', 'y']].values
    try:
        hull = ConvexHull(points)
        hull_pts = np.append(hull.vertices, hull.vertices[0])  # fermer le polygone

        # Zone remplie
        fig_zones.add_trace(go.Scatter(
            x=points[hull_pts, 0], y=points[hull_pts, 1],
            mode='lines',
            fill='toself',
            fillcolor=fill_color,
            line=dict(color=line_color, width=2, dash='dot'),
            name=ZONE_LABELS[cat],
            hoverinfo='skip',
        ))

        # Label au centroïde
        cx, cy = points[:, 0].mean(), points[:, 1].mean()
        fig_zones.add_annotation(
            x=cx, y=cy,
            text=ZONE_LABELS[cat],
            showarrow=False,
            font=dict(size=13, color=line_color, family='Arial Black'),
            bgcolor='rgba(30,30,30,0.7)',
            bordercolor=line_color,
            borderwidth=1,
            borderpad=4,
        )
    except Exception as e:
        print(f"[WARN] ConvexHull impossible pour {cat}: {e}")

# ─── B) Points ontologie (petits, par catégorie) ─────────────
for cat, (line_color, _) in ZONE_COLORS.items():
    mask = df_plot_onto['categorie'] == cat
    subset = df_plot_onto[mask]
    if len(subset) == 0:
        continue
    fig_zones.add_trace(go.Scatter(
        x=subset['x'], y=subset['y'],
        mode='markers',
        marker=dict(size=4, color=line_color, opacity=0.5),
        name=f"  {cat.replace('_',' ').title()}",
        text=subset['label'],
        customdata=subset['concept_id'].values,
        hovertemplate="<b>%{text}</b><br>ID: %{customdata}<extra></extra>",
        showlegend=False,
    ))

# ─── C) Étudiants (tous, étoiles grises) ─────────────────────
fig_zones.add_trace(go.Scatter(
    x=df_plot_students['x'], y=df_plot_students['y'],
    mode='markers',
    marker=dict(size=8, symbol='star', color='#FFD700', opacity=0.7,
                line=dict(width=0.8, color='#B8860B')),
    name='⭐ Termes étudiants',
    text=df_plot_students['label'],
    customdata=np.stack([df_plot_students['student_code'], df_plot_students['cas']], axis=-1),
    hovertemplate="<b>⭐ %{text}</b><br>Étudiant: %{customdata[0]}<br>Cas: %{customdata[1]}<extra></extra>",
))

# ─── D) Layout ────────────────────────────────────────────────
fig_zones.update_layout(
    title=dict(
        text="🏗️ Territoires Sémantiques de l'Ontologie ECG",
        font=dict(size=17, color='#e0e0e0'), x=0.5,
    ),
    xaxis=dict(title=f"{REDUCER_NAME} Dim 1", showgrid=True, gridcolor='#333', zeroline=False),
    yaxis=dict(title=f"{REDUCER_NAME} Dim 2", showgrid=True, gridcolor='#333', zeroline=False),
    plot_bgcolor='#1e1e1e', paper_bgcolor='#1e1e1e',
    font=dict(color='#e0e0e0'),
    legend=dict(bgcolor='#2d2d2d', bordercolor='#444', borderwidth=1, font=dict(size=11)),
    width=1200, height=800, hovermode='closest',
)

fig_zones.show()

print(f"\n🏗️ Carte des zones sémantiques tracée.")
print(f"   Les enveloppes convexes montrent les territoires de chaque catégorie ontologique.")
print(f"   Les ⭐ dorées montrent où se situent les termes étudiants par rapport à ces zones.")


🏗️ Carte des zones sémantiques tracée.
   Les enveloppes convexes montrent les territoires de chaque catégorie ontologique.
   Les ⭐ dorées montrent où se situent les termes étudiants par rapport à ces zones.


In [ ]:
# ============================================================
# CELLULE 7 — Lignes de Distance : Étudiant → Concept le + proche
# ============================================================
# Similarité cosinus dans l'espace ORIGINAL 1536D,
# lignes tracées dans l'espace UMAP 2D.

from sklearn.metrics.pairwise import cosine_similarity

# ─── A) Similarité cosinus étudiants × ontologie ─────────────
sim_matrix = cosine_similarity(embeddings_students, embeddings_onto)
print(f"[SIM] Matrice similarité : {sim_matrix.shape}")

nn_indices = sim_matrix.argmax(axis=1)
nn_scores  = sim_matrix.max(axis=1)
nn_distances = 1.0 - nn_scores

# ─── B) Tableau de correspondances ───────────────────────────
df_nn = pd.DataFrame({
    'terme_etudiant':     df_terms['terme'].values,
    'student_code':       df_terms['student_code'].values,
    'cas':                df_terms['cas'].values,
    'concept_onto':       [df_onto.iloc[i]['surface_form'] for i in nn_indices],
    'ontology_id':        [df_onto.iloc[i]['ontology_id'] for i in nn_indices],
    'categorie':          [df_onto.iloc[i]['categorie'] for i in nn_indices],
    'cosine_similarity':  nn_scores,
    'distance':           nn_distances,
    'etu_x': coords_students[:, 0],
    'etu_y': coords_students[:, 1],
    'onto_x': coords_onto[nn_indices, 0],
    'onto_y': coords_onto[nn_indices, 1],
})

# Stats
print(f"\n📊 Statistiques de distance (1 - cosine similarity) :")
print(f"   Moyenne   : {nn_distances.mean():.4f}")
print(f"   Médiane   : {np.median(nn_distances):.4f}")
print(f"   Min       : {nn_distances.min():.4f}  ({df_nn.iloc[nn_distances.argmin()]['terme_etudiant']} → {df_nn.iloc[nn_distances.argmin()]['concept_onto']})")
print(f"   Max       : {nn_distances.max():.4f}  ({df_nn.iloc[nn_distances.argmax()]['terme_etudiant']} → {df_nn.iloc[nn_distances.argmax()]['concept_onto']})")

n_excellent = (nn_scores >= 0.85).sum()
n_good      = ((nn_scores >= 0.70) & (nn_scores < 0.85)).sum()
n_medium    = ((nn_scores >= 0.50) & (nn_scores < 0.70)).sum()
n_poor      = (nn_scores < 0.50).sum()
total = len(nn_scores)
print(f"\n   🟢 Excellent (sim ≥ 0.85) : {n_excellent} ({100*n_excellent/total:.0f}%)")
print(f"   🟡 Bon      (0.70–0.85)   : {n_good} ({100*n_good/total:.0f}%)")
print(f"   🟠 Moyen    (0.50–0.70)   : {n_medium} ({100*n_medium/total:.0f}%)")
print(f"   🔴 Faible   (sim < 0.50)  : {n_poor} ({100*n_poor/total:.0f}%)")

# ─── C) Carte avec lignes — SEULEMENT les mauvais matchs ─────
# (sim < 0.75 = les lignes intéressantes, pas le bruit des matchs parfaits)
THRESHOLD_SHOW_LINE = 0.75
df_bad = df_nn[df_nn['cosine_similarity'] < THRESHOLD_SHOW_LINE]

fig2 = go.Figure()

# C.1) Fond ontologie gris
fig2.add_trace(go.Scatter(
    x=df_plot_onto['x'], y=df_plot_onto['y'],
    mode='markers',
    marker=dict(size=3, color='#555', opacity=0.25),
    name='Ontologie (fond)',
    text=df_plot_onto['label'],
    hovertemplate="<b>%{text}</b><extra>Ontologie</extra>",
))

# C.2) Lignes uniquement pour les mauvais matchs
for _, row in df_bad.iterrows():
    sim = row['cosine_similarity']
    t = max(0, min(1, (0.75 - sim) / 0.45))  # 0.75→0 (jaune), 0.30→1 (rouge)
    r = 255
    g = int((1 - t) * 200)
    color = f'rgb({r},{g},0)'

    fig2.add_trace(go.Scatter(
        x=[row['etu_x'], row['onto_x']],
        y=[row['etu_y'], row['onto_y']],
        mode='lines',
        line=dict(color=color, width=1.5),
        opacity=0.5,
        showlegend=False,
        hoverinfo='skip',
    ))

# C.3) Étoiles colorées par similarité (tous les étudiants)
fig2.add_trace(go.Scatter(
    x=df_nn['etu_x'], y=df_nn['etu_y'],
    mode='markers',
    marker=dict(
        size=9, symbol='star',
        color=df_nn['cosine_similarity'],
        colorscale=[[0, '#F44336'], [0.5, '#FF9800'], [0.75, '#FFEB3B'], [1, '#4CAF50']],
        cmin=0.3, cmax=1.0,
        line=dict(width=0.8, color='#333'),
        colorbar=dict(
            title=dict(text='Similarité<br>cosinus', font=dict(size=11)),
            thickness=15, len=0.6,
            tickvals=[0.4, 0.6, 0.8, 1.0],
        ),
    ),
    name='⭐ Termes étudiants',
    text=df_nn['terme_etudiant'],
    customdata=np.stack([
        df_nn['concept_onto'],
        df_nn['cosine_similarity'].round(3).astype(str),
        df_nn['student_code'],
    ], axis=-1),
    hovertemplate=(
        "<b>⭐ %{text}</b><br>"
        "→ <b>%{customdata[0]}</b> (sim: %{customdata[1]})<br>"
        "Étudiant: %{customdata[2]}<extra></extra>"
    ),
))

fig2.update_layout(
    title=dict(
        text=f"🎯 Distance Sémantique — Lignes pour matchs faibles (sim < {THRESHOLD_SHOW_LINE})",
        font=dict(size=16, color='#e0e0e0'), x=0.5,
    ),
    xaxis=dict(title=f"{REDUCER_NAME} Dim 1", showgrid=True, gridcolor='#333', zeroline=False),
    yaxis=dict(title=f"{REDUCER_NAME} Dim 2", showgrid=True, gridcolor='#333', zeroline=False),
    plot_bgcolor='#1e1e1e', paper_bgcolor='#1e1e1e',
    font=dict(color='#e0e0e0'),
    legend=dict(bgcolor='#2d2d2d', bordercolor='#444', borderwidth=1, font=dict(size=11)),
    width=1200, height=800, hovermode='closest',
)

fig2.show()

print(f"\n🎯 Carte des distances tracée.")
print(f"   Lignes affichées : {len(df_bad)} termes (sim < {THRESHOLD_SHOW_LINE})")
print(f"   🟢 Bons matchs (pas de ligne) : {len(df_nn) - len(df_bad)}")
print(f"   🟠→? Les lignes longues/rouges = termes mal couverts par l'ontologie")

In [ ]:
# ============================================================
# CELLULE 8 — Tableau Détaillé + Histogramme
# ============================================================

df_display = df_nn[['student_code', 'terme_etudiant', 'concept_onto', 'ontology_id',
                     'categorie', 'cosine_similarity']].copy()
df_display = df_display.rename(columns={
    'student_code':      '👤 Étudiant',
    'terme_etudiant':    '⭐ Terme',
    'concept_onto':      '🔵 Concept Ontologie',
    'ontology_id':       'ID OWL',
    'categorie':         'Catégorie',
    'cosine_similarity': 'Sim.',
})
df_display['Sim.'] = df_display['Sim.'].round(4)
df_sorted = df_display.sort_values('Sim.', ascending=True)

print("=" * 90)
print("🔴 TOP 30 — Termes étudiants les PLUS ÉLOIGNÉS de l'ontologie")
print("   (= termes mal couverts, potentiels trous dans l'ontologie)")
print("=" * 90)
display(df_sorted.head(30).reset_index(drop=True))

print("\n" + "=" * 90)
print("🟢 TOP 15 — Termes étudiants les PLUS PROCHES de l'ontologie")
print("=" * 90)
display(df_sorted.tail(15).sort_values('Sim.', ascending=False).reset_index(drop=True))

# ─── Histogramme de distribution ──────────────────────────────
fig3 = go.Figure()
fig3.add_trace(go.Histogram(
    x=df_nn['cosine_similarity'],
    nbinsx=30,
    marker_color='#2196F3',
    opacity=0.8,
))

for thresh, label, color in [(0.85, 'Excellent', '#4CAF50'), (0.70, 'Bon', '#FF9800'), (0.50, 'Moyen', '#F44336')]:
    fig3.add_vline(x=thresh, line=dict(color=color, dash='dash', width=2),
                   annotation_text=label, annotation_position="top")

fig3.update_layout(
    title=dict(text="📊 Distribution des Similarités Cosinus (Étudiant → Concept le + proche)",
               font=dict(size=14, color='#e0e0e0'), x=0.5),
    xaxis=dict(title="Similarité Cosinus", range=[0.2, 1.05]),
    yaxis=dict(title="Nombre de termes"),
    plot_bgcolor='#1e1e1e', paper_bgcolor='#1e1e1e',
    font=dict(color='#e0e0e0'),
    width=900, height=400, bargap=0.05,
)
fig3.show()

# ─── Stats par étudiant ──────────────────────────────────────
print(f"\n📊 Score moyen de similarité par étudiant :")
for code, grp in df_nn.groupby('student_code'):
    mean_sim = grp['cosine_similarity'].mean()
    n_bad = (grp['cosine_similarity'] < 0.70).sum()
    bar = '█' * int(mean_sim * 30) + '░' * (30 - int(mean_sim * 30))
    print(f"   {code} : {bar} {mean_sim:.3f}  ({n_bad} termes < 0.70)")

print(f"\n💡 Les termes 🔴 en haut du tableau sont des candidats pour enrichir l'ontologie.")

🔴 TOP 20 — Termes étudiants les PLUS ÉLOIGNÉS de l'ontologie
   (= termes mal couverts, potentiels trous dans l'ontologie)


,⭐ Terme Étudiant,🔵 Concept Ontologie,ID OWL,Catégorie,Sim. Cosinus
0,ample,Apex,APEX,DESCRIPTEUR_ECG,0.3967
1,62 bpm,Tachycardie,TACHYCARDIE,DESCRIPTEUR_ECG,0.4133
2,assez fines,ST moins,COURANT_DE_LÉSION_SOUS_ENDOCARDIQUE,SIGNE_ECG_PATHOLOGIQUE,0.4137
3,78 bpm,Tachycardie,TACHYCARDIE,DESCRIPTEUR_ECG,0.4157
4,62bpm,Rythme sinusal,RYTHME_SINUSAL,SIGNE_ECG_PATHOLOGIQUE,0.4446
5,HAG limite,HVG,HYPERTROPHIE_VENTRICULAIRE_GAUCHE,DIAGNOSTIC_MAJEUR,0.4630
6,75 bpm,Tachycardie,TACHYCARDIE,DESCRIPTEUR_ECG,0.4720
7,symétrique,TJ orthodromique utilisant une voie accessoire,TJ_ORTHODROMIQUE_UTILISANT_UNE_VOIE_ACCESSOIRE,DIAGNOSTIC_MAJEUR,0.4727
8,conduction 4/1,Faisceau accessoire à conduction antérograde,FAISCEAU_ACCESSOIRE_À_CONDUCTION_ANTÉROGRADE,DIAGNOSTIC_MAJEUR,0.4766
9,70 bpm,Tachycardie,TACHYCARDIE,DESCRIPTEUR_ECG,0.4852



🟢 TOP 20 — Termes étudiants les PLUS PROCHES de l'ontologie
   (= excellent alignement sémantique)


,⭐ Terme Étudiant,🔵 Concept Ontologie,ID OWL,Catégorie,Sim. Cosinus
0,BAV1,BAV1,BAV_DE_TYPE_1,DESCRIPTEUR_ECG,1.00
1,QRS fins,QRS fins,QRS_FINS,DESCRIPTEUR_ECG,1.00
2,Rythme atrial électroentrainé,Rythme atrial électroentrainé,STIMULATION_ATRIALE,DESCRIPTEUR_ECG,1.00
3,Rythme sinusal,Rythme sinusal,RYTHME_SINUSAL,SIGNE_ECG_PATHOLOGIQUE,1.00
4,Bloc de branche gauche,Bloc de branche gauche,BLOC_DE_BRANCHE_GAUCHE,SIGNE_ECG_PATHOLOGIQUE,1.00
5,BAV complet,BAV complet,BAV_COMPLET,DIAGNOSTIC_URGENT,1.00
6,BAV 1,BAV 1,BAV_DE_TYPE_1,DESCRIPTEUR_ECG,1.00
7,Axe normal,Axe normal,AXE_NORMAL,DESCRIPTEUR_ECG,1.00
8,BBD complet,BBD complet,BLOC_DE_BRANCHE_DROIT_COMPLET,DIAGNOSTIC_MAJEUR,1.00
9,BBG,BBG,BLOC_DE_BRANCHE_GAUCHE,SIGNE_ECG_PATHOLOGIQUE,1.00



💡 Les termes 🔴 en bas du tableau sont des candidats pour enrichir l'ontologie.
